# MobileNetV2 calibrated benchmark on the locked QGECAF patient folds

This runner trains ten MobileNetV2 models, fits fold-specific temperature scaling using only inner-validation logits, and compares the calibrated results with the completed QGECAF experiment. The untouched final-test cohort is not evaluated.

Place this notebook and `MobileNetV2_Calibrated_10Fold_Automated.py` in the same folder. Select the `chaitalienv` kernel and run the cells in order. If an earlier run failed with `UnsortedSegmentSum`, restart the kernel before running this corrected notebook.

In [1]:
import os
os.environ['TF_DETERMINISTIC_OPS'] = '0'

from pathlib import Path
import pandas as pd
from MobileNetV2_Calibrated_10Fold_Automated import Config, run
import tensorflow as tf

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.10.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
cfg = Config(
    qgecaf_root=(
        r'D:\chaitali\qgacef'
        r'\QGECAF_REVIEWER_REVISION_NESTED_BGWO'
    ),
    output=(
        r'D:\chaitali\qgacef'
        r'\MOBILENETV2_CALIBRATED_SAME_PATIENT_FOLDS'
    ),
    image_size=128,
    num_classes=6,
    outer_folds=10,
    seed=42,
    batch_size=8,
    epochs=50,
    learning_rate=1e-4,
    fine_tune_last=30,
    dropout=0.30,
    early_stopping_patience=8,
    resume=True,
)
cfg

Config(qgecaf_root='D:\\chaitali\\qgacef\\QGECAF_REVIEWER_REVISION_NESTED_BGWO', output='D:\\chaitali\\qgacef\\MOBILENETV2_CALIBRATED_SAME_PATIENT_FOLDS', image_size=128, num_classes=6, outer_folds=10, seed=42, batch_size=8, epochs=50, learning_rate=0.0001, fine_tune_last=30, dropout=0.3, early_stopping_patience=8, temperature_min=0.05, temperature_max=100.0, ece_bins=15, bootstrap_repetitions=10000, resume=True, smoke_test=False)

In [3]:
reference_root = Path(cfg.qgecaf_root)
required = [
    reference_root / 'locked_patient_fold_assignments.csv',
    reference_root / 'fold_metrics.csv',
    reference_root / 'development_out_of_fold_predictions.csv',
]
for path in required:
    print(path, '->', path.is_file())
    assert path.is_file(), f'Missing required file: {path}'

for fold in range(1, 11):
    audit = (reference_root / 'cross_validation' / f'fold_{fold:02d}'
             / 'patient_partition_audit.csv')
    assert audit.is_file(), f'Missing fold audit: {audit}'

print('PASS: Reference metrics, locked manifest and all ten partition audits exist.')

D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\locked_patient_fold_assignments.csv -> True
D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\fold_metrics.csv -> True
D:\chaitali\qgacef\QGECAF_REVIEWER_REVISION_NESTED_BGWO\development_out_of_fold_predictions.csv -> True
PASS: Reference metrics, locked manifest and all ten partition audits exist.


## Run or safely resume

This is the only long-running cell. Re-running it with `resume=True` verifies completed folds and skips them. Do not change the output path or protocol settings during a run.

In [4]:
run(cfg)

TensorFlow 2.10.0; GPUs detected: 1
Reproducibility: fixed seeds and deterministic input ordering; strict GPU-op determinism disabled for TensorFlow 2.10 compatibility.

MOBILENETV2 OUTER FOLD 1/10
Instructions for updating:
Use `tf.compat.v1.graph_util.tensor_shape_from_node_def_name`
Epoch 1/50
1143/1143 [==============================] - ETA: 0s - loss: 0.3392 - accuracy: 0.8145
Epoch 1: val_loss improved from inf to 0.53334, saving model to D:\chaitali\qgacef\MOBILENETV2_CALIBRATED_SAME_PATIENT_FOLDS\cross_validation\fold_01\best_model.keras
1143/1143 [==============================] - 41s 27ms/step - loss: 0.3392 - accuracy: 0.8145 - val_loss: 0.5333 - val_accuracy: 0.8246 - lr: 1.0000e-04
Epoch 2/50
1143/1143 [==============================] - ETA: 0s - loss: 0.1745 - accuracy: 0.8972
Epoch 2: val_loss improved from 0.53334 to 0.35448, saving model to D:\chaitali\qgacef\MOBILENETV2_CALIBRATED_SAME_PATIENT_FOLDS\cross_validation\fold_01\best_model.keras
1143/1143 [================

In [5]:
result_root = Path(cfg.output)
display(pd.read_csv(result_root / 'summary_uncalibrated_mean_sd_ci95.csv'))
display(pd.read_csv(result_root / 'summary_temperature_scaled_mean_sd_ci95.csv'))
display(pd.read_csv(result_root / 'paired_qgecaf_vs_calibrated_mobilenetv2.csv'))
print('Run complete:', (result_root / 'RUN_COMPLETE.json').is_file())

,metric,folds,mean,standard_deviation,ci95_low,ci95_high
0,accuracy,10,0.928318,0.009659,0.921408,0.935228
1,balanced_accuracy,10,0.923563,0.011339,0.915451,0.931674
2,macro_precision,10,0.905329,0.015689,0.894106,0.916553
3,macro_recall,10,0.923563,0.011339,0.915451,0.931674
4,macro_f1,10,0.912988,0.010600,0.905405,0.920571
5,weighted_f1,10,0.928672,0.009614,0.921795,0.935550
6,cohen_kappa,10,0.907742,0.011492,0.899522,0.915963
7,mcc,10,0.908132,0.011325,0.900031,0.916234
8,macro_specificity,10,0.984970,0.001722,0.983738,0.986202
9,ece_15_bins,10,0.038430,0.009482,0.031648,0.045213


,metric,folds,mean,standard_deviation,ci95_low,ci95_high
0,accuracy,10,0.928318,0.009659,0.921408,0.935228
1,balanced_accuracy,10,0.923563,0.011339,0.915451,0.931674
2,macro_precision,10,0.905329,0.015689,0.894106,0.916553
3,macro_recall,10,0.923563,0.011339,0.915451,0.931674
4,macro_f1,10,0.912988,0.010600,0.905405,0.920571
5,weighted_f1,10,0.928672,0.009614,0.921795,0.935550
6,cohen_kappa,10,0.907742,0.011492,0.899522,0.915963
7,mcc,10,0.908132,0.011325,0.900031,0.916234
8,macro_specificity,10,0.984970,0.001722,0.983738,0.986202
9,ece_15_bins,10,0.022231,0.006070,0.017889,0.026573


,metric,qgecaf_mean,calibrated_mobilenetv2_mean,mean_difference_qgecaf_minus_baseline,paired_bootstrap_ci95_low,paired_bootstrap_ci95_high,wilcoxon_statistic,raw_p,rank_biserial_qgecaf_minus_baseline,preferred_direction,holm_adjusted_p,significant_at_0.05
0,macro_f1,0.911895,0.912988,-0.001093,-0.006679,0.004194,27.0,1.000000,-0.018182,higher,1.000000,False
1,negative_log_likelihood,0.298073,0.225949,0.072125,0.058576,0.086032,0.0,0.001953,1.000000,lower,0.007812,True
2,brier_multiclass,0.121129,0.111642,0.009487,0.003729,0.015365,6.0,0.027344,0.781818,lower,0.082031,False
3,ece_15_bins,0.029862,0.022231,0.007631,0.000540,0.014541,9.0,0.064453,0.672727,lower,0.128906,False


Run complete: True
